# Module 08: Encoder-Decoder (Seq2Seq)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/08-encoder-decoder/notebook.ipynb)

**GPU recommended:** No (toy task trains in under 2 minutes on CPU).

## Setup and Imports

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

plt.style.use("seaborn-v0_8-whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 1. Core Building Blocks

We start with the primitives shared by encoder and decoder:
scaled dot-product attention, multi-head attention, position-wise
feed-forward networks, and sinusoidal positional encodings.

### 1.1 Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """Compute scaled dot-product attention.

    Args:
        query: (batch, ..., seq_len_q, d_k)
        key:   (batch, ..., seq_len_k, d_k)
        value: (batch, ..., seq_len_k, d_v)
        mask:  broadcastable to (batch, ..., seq_len_q, seq_len_k)

    Returns:
        output:  (batch, ..., seq_len_q, d_v)
        weights: (batch, ..., seq_len_q, seq_len_k)
    """
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, value)
    return output, weights

### 1.2 Multi-Head Attention

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W^O$$

This module is used three different ways in an encoder-decoder transformer:
- **Encoder self-attention:** Q=K=V come from the encoder (bidirectional).
- **Decoder self-attention:** Q=K=V come from the decoder (causal mask).
- **Cross-attention:** Q comes from the decoder, K=V come from the encoder.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

        self.attn_weights = None  # stored for visualization

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        q = self.w_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        if mask is not None and mask.dim() == 3:
            mask = mask.unsqueeze(1)  # (batch, 1, seq_q, seq_k)

        output, weights = scaled_dot_product_attention(q, k, v, mask=mask)
        self.attn_weights = weights.detach()

        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.w_o(output)

### 1.3 Position-wise Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

### 1.4 Sinusoidal Positional Encoding

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

---
## 2. Encoder: Bidirectional Self-Attention

Each encoder layer applies bidirectional (unmasked) self-attention followed
by a feed-forward network, both wrapped with residual connections and
layer normalization (Pre-LN variant).

```
x -> LayerNorm -> Self-Attention(Q=x, K=x, V=x) -> + residual
                                                     |
                               LayerNorm -> FFN   -> + residual
```

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        normed = self.ln1(x)
        x = x + self.dropout1(self.self_attn(normed, normed, normed, mask=src_mask))
        normed = self.ln2(x)
        x = x + self.dropout2(self.ff(normed))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None):
        """Encode the source sequence.

        Args:
            src: (batch, src_len) token indices
            src_mask: (batch, 1, src_len) padding mask

        Returns:
            (batch, src_len, d_model) encoder representations
        """
        x = self.pos_enc(self.token_emb(src) * math.sqrt(self.d_model))
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.ln_final(x)

### Verify encoder shapes

In [ ]:
torch.manual_seed(42)

enc = Encoder(vocab_size=20, d_model=64, num_heads=4, d_ff=256, num_layers=2)
dummy_src = torch.randint(0, 20, (2, 8))
enc_out = enc(dummy_src)

print(f"Source shape:  {dummy_src.shape}")
print(f"Encoder output shape: {enc_out.shape}")
assert enc_out.shape == (2, 8, 64)
print("Encoder shape check passed.")

---
## 3. Decoder: Causal Self-Attention + Cross-Attention

Each decoder layer has three sub-layers:
1. **Causal self-attention** over the target sequence (masked so each position can only attend to earlier positions).
2. **Cross-attention** where the decoder queries attend to encoder key-value pairs.
3. **Feed-forward network.**

```
y -> LayerNorm -> Causal Self-Attn(Q=y, K=y, V=y) -> + residual
                                                       |
     LayerNorm -> Cross-Attn(Q=y, K=enc, V=enc)    -> + residual
                                                       |
                          LayerNorm -> FFN           -> + residual
```

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # Causal self-attention
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.dropout1 = nn.Dropout(dropout)

        # Cross-attention
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.dropout2 = nn.Dropout(dropout)

        # Feed-forward
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None, src_mask=None):
        # 1. Causal self-attention
        normed = self.ln1(x)
        x = x + self.dropout1(self.self_attn(normed, normed, normed, mask=tgt_mask))

        # 2. Cross-attention to encoder output
        normed = self.ln2(x)
        x = x + self.dropout2(self.cross_attn(normed, enc_output, enc_output, mask=src_mask))

        # 3. Feed-forward
        normed = self.ln3(x)
        x = x + self.dropout3(self.ff(normed))
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)

    def forward(self, tgt, enc_output, tgt_mask=None, src_mask=None):
        """Decode the target sequence given encoder output.

        Args:
            tgt: (batch, tgt_len) token indices
            enc_output: (batch, src_len, d_model)
            tgt_mask: (batch, tgt_len, tgt_len) causal + padding mask
            src_mask: (batch, 1, src_len) source padding mask

        Returns:
            (batch, tgt_len, d_model) decoder representations
        """
        x = self.pos_enc(self.token_emb(tgt) * math.sqrt(self.d_model))
        for layer in self.layers:
            x = layer(x, enc_output, tgt_mask, src_mask)
        return self.ln_final(x)

### Verify decoder shapes

In [ ]:
torch.manual_seed(42)

dec = Decoder(vocab_size=20, d_model=64, num_heads=4, d_ff=256, num_layers=2)
dummy_tgt = torch.randint(0, 20, (2, 6))
# enc_out from the encoder test above: (2, 8, 64)

# Causal mask for the target
tgt_len = dummy_tgt.size(1)
causal_mask = torch.tril(torch.ones(tgt_len, tgt_len)).unsqueeze(0)

dec_out = dec(dummy_tgt, enc_out, tgt_mask=causal_mask)

print(f"Target shape:         {dummy_tgt.shape}")
print(f"Encoder output shape: {enc_out.shape}")
print(f"Decoder output shape: {dec_out.shape}")
assert dec_out.shape == (2, 6, 64)
print("Decoder shape check passed.")

---
## 4. Full Encoder-Decoder Transformer

The complete model wires the encoder, decoder, and a linear projection
head together. It also provides helper methods for creating the causal
and padding masks.

In [ ]:
class EncoderDecoderTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads,
                 d_ff, num_encoder_layers, num_decoder_layers,
                 max_len=512, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx

        self.encoder = Encoder(
            src_vocab_size, d_model, num_heads, d_ff,
            num_encoder_layers, max_len, dropout
        )
        self.decoder = Decoder(
            tgt_vocab_size, d_model, num_heads, d_ff,
            num_decoder_layers, max_len, dropout
        )
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)

        # Tie decoder embedding weights with output projection
        self.output_proj.weight = self.decoder.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def make_src_mask(self, src):
        """Create source padding mask: (batch, 1, src_len)."""
        return (src != self.pad_idx).unsqueeze(1)

    def make_tgt_mask(self, tgt):
        """Create target mask combining causal mask with padding mask.

        Returns: (batch, tgt_len, tgt_len)
        """
        tgt_len = tgt.size(1)
        causal = torch.tril(torch.ones(tgt_len, tgt_len, device=tgt.device)).unsqueeze(0)
        padding = (tgt != self.pad_idx).unsqueeze(1)  # (batch, 1, tgt_len)
        return causal & padding

    def forward(self, src, tgt):
        """Full forward pass.

        Args:
            src: (batch, src_len)
            tgt: (batch, tgt_len)

        Returns:
            logits: (batch, tgt_len, tgt_vocab_size)
        """
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)

        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, tgt_mask, src_mask)
        logits = self.output_proj(dec_output)
        return logits

    def encode(self, src):
        """Encode source (used during inference)."""
        src_mask = self.make_src_mask(src)
        return self.encoder(src, src_mask), src_mask

    def decode_step(self, tgt, enc_output, src_mask):
        """One decoding step (used during inference)."""
        tgt_mask = self.make_tgt_mask(tgt)
        dec_output = self.decoder(tgt, enc_output, tgt_mask, src_mask)
        logits = self.output_proj(dec_output)
        return logits

### Verify the full model

In [ ]:
torch.manual_seed(42)

model = EncoderDecoderTransformer(
    src_vocab_size=20,
    tgt_vocab_size=20,
    d_model=64,
    num_heads=4,
    d_ff=256,
    num_encoder_layers=2,
    num_decoder_layers=2,
    pad_idx=0,
)

dummy_src = torch.randint(1, 20, (2, 8))
dummy_tgt = torch.randint(1, 20, (2, 6))

logits = model(dummy_src, dummy_tgt)
print(f"Source shape:  {dummy_src.shape}")
print(f"Target shape:  {dummy_tgt.shape}")
print(f"Logits shape:  {logits.shape}")
assert logits.shape == (2, 6, 20)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print("Full encoder-decoder shape check passed.")

---
## 5. Toy Seq2Seq Task: Number to Words

We train the model to convert digit strings into English word sequences.
For example:

```
Source: 1 4 0 5
Target: [BOS] one four zero five [EOS]
```

This is a genuine sequence-to-sequence task: the input and output
vocabularies are different, and the output length differs from the input.

### 5.1 Vocabulary and dataset

In [ ]:
# Source vocabulary: digits + PAD
SRC_PAD = 0
# Digits are 1-10 (digit 0 maps to token 1, digit 9 maps to token 10)
SRC_VOCAB_SIZE = 11  # PAD + 10 digits

# Target vocabulary: word tokens + special tokens
TGT_PAD = 0
TGT_BOS = 1
TGT_EOS = 2
# Word tokens start at 3
WORD_TOKENS = {
    0: 3,   # "zero"
    1: 4,   # "one"
    2: 5,   # "two"
    3: 6,   # "three"
    4: 7,   # "four"
    5: 8,   # "five"
    6: 9,   # "six"
    7: 10,  # "seven"
    8: 11,  # "eight"
    9: 12,  # "nine"
}
TGT_VOCAB_SIZE = 13  # PAD + BOS + EOS + 10 word tokens

TGT_TOKEN_NAMES = {
    TGT_PAD: "[PAD]", TGT_BOS: "[BOS]", TGT_EOS: "[EOS]",
    3: "zero", 4: "one", 5: "two", 6: "three", 7: "four",
    8: "five", 9: "six", 10: "seven", 11: "eight", 12: "nine",
}
SRC_TOKEN_NAMES = {SRC_PAD: "[PAD]"}
SRC_TOKEN_NAMES.update({d + 1: str(d) for d in range(10)})


def src_tokens_to_str(tokens):
    return " ".join(SRC_TOKEN_NAMES.get(t, f"?{t}") for t in tokens if t != SRC_PAD)


def tgt_tokens_to_str(tokens):
    return " ".join(TGT_TOKEN_NAMES.get(t, f"?{t}") for t in tokens if t != TGT_PAD)


print(f"Source vocab size: {SRC_VOCAB_SIZE}")
print(f"Target vocab size: {TGT_VOCAB_SIZE}")
print(f"Source tokens: {SRC_TOKEN_NAMES}")
print(f"Target tokens: {TGT_TOKEN_NAMES}")

In [ ]:
def generate_number_to_words_dataset(num_samples, min_digits=1, max_digits=5):
    """Generate number-to-words dataset.

    Source: digit tokens (digit_value + 1, since 0 is PAD), padded to max_digits.
    Target input: [BOS] + word tokens, padded to max_digits + 1.
    Target label: word tokens + [EOS], padded to max_digits + 1 (with -100 for padding).
    """
    src_max_len = max_digits
    tgt_max_len = max_digits + 2  # BOS + words + EOS

    all_src = []
    all_tgt_in = []   # decoder input: BOS + words
    all_tgt_out = []  # decoder target: words + EOS

    for _ in range(num_samples):
        length = torch.randint(min_digits, max_digits + 1, (1,)).item()
        digits = torch.randint(0, 10, (length,)).tolist()

        # Source: digit + 1 (to avoid PAD=0)
        src = [d + 1 for d in digits]
        src_padded = src + [SRC_PAD] * (src_max_len - len(src))

        # Target: word tokens
        words = [WORD_TOKENS[d] for d in digits]
        tgt_in = [TGT_BOS] + words
        tgt_out = words + [TGT_EOS]

        # Pad target
        tgt_in_padded = tgt_in + [TGT_PAD] * (tgt_max_len - len(tgt_in))
        tgt_out_padded = tgt_out + [-100] * (tgt_max_len - len(tgt_out))

        all_src.append(src_padded)
        all_tgt_in.append(tgt_in_padded)
        all_tgt_out.append(tgt_out_padded)

    return (
        torch.tensor(all_src),
        torch.tensor(all_tgt_in),
        torch.tensor(all_tgt_out),
    )

In [ ]:
torch.manual_seed(42)

train_src, train_tgt_in, train_tgt_out = generate_number_to_words_dataset(4000)
val_src, val_tgt_in, val_tgt_out = generate_number_to_words_dataset(500)

print(f"Train: src={train_src.shape}, tgt_in={train_tgt_in.shape}, tgt_out={train_tgt_out.shape}")
print(f"Val:   src={val_src.shape}, tgt_in={val_tgt_in.shape}, tgt_out={val_tgt_out.shape}")
print()

for i in range(5):
    print(f"Source:    {src_tokens_to_str(train_src[i].tolist())}")
    print(f"Tgt input: {tgt_tokens_to_str(train_tgt_in[i].tolist())}")
    tgt_labels = [TGT_TOKEN_NAMES.get(t, "_") if t != -100 else "_" for t in train_tgt_out[i].tolist()]
    print(f"Tgt label: {' '.join(tgt_labels)}")
    print()

### 5.2 Training loop

In [ ]:
torch.manual_seed(42)

D_MODEL = 64
NUM_HEADS = 4
D_FF = 256
NUM_LAYERS = 3
DROPOUT = 0.1
BATCH_SIZE = 64
NUM_EPOCHS = 40
LR = 3e-4

model = EncoderDecoderTransformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_encoder_layers=NUM_LAYERS,
    num_decoder_layers=NUM_LAYERS,
    pad_idx=0,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Training samples: {len(train_src)}")

In [ ]:
train_src_dev = train_src.to(device)
train_tgt_in_dev = train_tgt_in.to(device)
train_tgt_out_dev = train_tgt_out.to(device)
val_src_dev = val_src.to(device)
val_tgt_in_dev = val_tgt_in.to(device)
val_tgt_out_dev = val_tgt_out.to(device)

train_losses = []
val_losses = []
val_accuracies = []

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    perm = torch.randperm(len(train_src_dev))
    src_shuffled = train_src_dev[perm]
    tgt_in_shuffled = train_tgt_in_dev[perm]
    tgt_out_shuffled = train_tgt_out_dev[perm]

    for i in range(0, len(train_src_dev), BATCH_SIZE):
        batch_src = src_shuffled[i:i + BATCH_SIZE]
        batch_tgt_in = tgt_in_shuffled[i:i + BATCH_SIZE]
        batch_tgt_out = tgt_out_shuffled[i:i + BATCH_SIZE]

        logits = model(batch_src, batch_tgt_in)

        loss = F.cross_entropy(
            logits.reshape(-1, TGT_VOCAB_SIZE),
            batch_tgt_out.reshape(-1),
            ignore_index=-100,
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = model(val_src_dev, val_tgt_in_dev)
        val_loss = F.cross_entropy(
            val_logits.reshape(-1, TGT_VOCAB_SIZE),
            val_tgt_out_dev.reshape(-1),
            ignore_index=-100,
        ).item()
        val_losses.append(val_loss)

        preds = val_logits.argmax(dim=-1)
        valid_mask = val_tgt_out_dev != -100
        correct = (preds == val_tgt_out_dev) & valid_mask
        accuracy = correct.sum().item() / valid_mask.sum().item()
        val_accuracies.append(accuracy)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        print(
            f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
            f"Train loss: {avg_train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val acc: {accuracy:.4f} | "
            f"Time: {elapsed:.1f}s"
        )

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time:.1f}s")
print(f"Final validation token accuracy: {val_accuracies[-1]:.4f}")

### 5.3 Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, train_losses, label="Train", linewidth=1.5)
ax1.plot(epochs_range, val_losses, label="Validation", linewidth=1.5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training and Validation Loss")
ax1.legend()

ax2.plot(epochs_range, val_accuracies, color="green", linewidth=1.5)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Token Accuracy")
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## 6. Teacher Forcing vs Autoregressive Generation

During training we use **teacher forcing**: at each decoder step the
ground-truth previous token is fed as input. At inference time, the model
must use its own predictions, which can cause **exposure bias** --
errors compound because the model never saw its own mistakes during training.

We compare both strategies here.

### 6.1 Greedy autoregressive decoding

In [ ]:
@torch.no_grad()
def greedy_decode(model, src, max_len=20):
    """Autoregressive greedy decoding.

    Args:
        model: EncoderDecoderTransformer
        src: (1, src_len) source token ids
        max_len: maximum number of tokens to generate

    Returns:
        List of generated token ids (excluding BOS)
    """
    model.eval()
    src = src.to(device)
    enc_output, src_mask = model.encode(src)

    tgt_tokens = [TGT_BOS]

    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_tokens], device=device)
        logits = model.decode_step(tgt_tensor, enc_output, src_mask)
        next_token = logits[0, -1].argmax().item()
        tgt_tokens.append(next_token)
        if next_token == TGT_EOS:
            break

    return tgt_tokens[1:]  # exclude BOS

In [ ]:
# Teacher forcing accuracy (using ground-truth inputs)
model.eval()
with torch.no_grad():
    val_logits_tf = model(val_src_dev, val_tgt_in_dev)
    preds_tf = val_logits_tf.argmax(dim=-1)
    valid_mask = val_tgt_out_dev != -100
    correct_tf = (preds_tf == val_tgt_out_dev) | ~valid_mask
    seq_correct_tf = correct_tf.all(dim=-1)
    seq_acc_tf = seq_correct_tf.float().mean().item()

# Autoregressive accuracy
num_correct_ar = 0
num_total_ar = len(val_src)

for i in range(num_total_ar):
    src_sample = val_src[i].unsqueeze(0)
    generated = greedy_decode(model, src_sample, max_len=10)

    # Expected output: word tokens + EOS
    expected = [t for t in val_tgt_out[i].tolist() if t != -100]
    if generated == expected:
        num_correct_ar += 1

seq_acc_ar = num_correct_ar / num_total_ar

print(f"Full-sequence accuracy (teacher forcing): {seq_acc_tf:.4f} ({seq_correct_tf.sum().item()}/{len(val_src)})")
print(f"Full-sequence accuracy (autoregressive):  {seq_acc_ar:.4f} ({num_correct_ar}/{num_total_ar})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

methods = ["Teacher Forcing", "Autoregressive"]
accs = [seq_acc_tf, seq_acc_ar]
colors = ["steelblue", "coral"]

bars = ax.bar(methods, accs, color=colors, width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.3f}", ha="center", va="bottom", fontsize=11)

ax.set_ylabel("Full-Sequence Accuracy")
ax.set_title("Teacher Forcing vs Autoregressive Decoding")
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

### 6.2 Qualitative examples

In [ ]:
test_numbers = [
    [3, 1, 4],
    [7, 2],
    [9, 0, 5, 8],
    [1, 2, 3, 4, 5],
    [6],
    [0, 0, 0],
]

print("Autoregressive decoding examples:")
print("-" * 65)
for digits in test_numbers:
    src_tokens = torch.tensor([[d + 1 for d in digits] + [SRC_PAD] * (5 - len(digits))])
    generated = greedy_decode(model, src_tokens, max_len=10)

    # Remove EOS for display
    gen_words = [TGT_TOKEN_NAMES.get(t, f"?{t}") for t in generated if t != TGT_EOS]
    expected_words = [TGT_TOKEN_NAMES[WORD_TOKENS[d]] for d in digits]

    match = gen_words == expected_words
    status = "PASS" if match else "FAIL"
    print(f"  Input: {digits}")
    print(f"  Expected: {' '.join(expected_words)}")
    print(f"  Got:      {' '.join(gen_words)}  [{status}]")
    print()

---
## 7. Beam Search

Greedy decoding picks the single most probable token at each step, which
can miss globally better sequences. **Beam search** maintains $k$ candidate
sequences (beams) and expands each, keeping only the top-$k$ by cumulative
log-probability.

In [ ]:
@torch.no_grad()
def beam_search(model, src, beam_width=3, max_len=20):
    """Beam search decoding.

    Args:
        model: EncoderDecoderTransformer
        src: (1, src_len) source token ids
        beam_width: number of beams to maintain
        max_len: maximum output length

    Returns:
        List of (log_prob, token_list) tuples sorted by score, best first.
    """
    model.eval()
    src = src.to(device)
    enc_output, src_mask = model.encode(src)

    # Each beam: (cumulative_log_prob, token_list)
    beams = [(0.0, [TGT_BOS])]
    completed = []

    for _ in range(max_len):
        candidates = []

        for log_prob, tokens in beams:
            if tokens[-1] == TGT_EOS:
                completed.append((log_prob, tokens))
                continue

            tgt_tensor = torch.tensor([tokens], device=device)
            logits = model.decode_step(tgt_tensor, enc_output, src_mask)
            log_probs = F.log_softmax(logits[0, -1], dim=-1)

            top_log_probs, top_indices = log_probs.topk(beam_width)

            for lp, idx in zip(top_log_probs.tolist(), top_indices.tolist()):
                candidates.append((log_prob + lp, tokens + [idx]))

        if not candidates:
            break

        # Keep top beam_width candidates
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]

    # Add any remaining beams to completed
    completed.extend(beams)
    completed.sort(key=lambda x: x[0], reverse=True)

    # Return tokens without BOS
    return [(lp, toks[1:]) for lp, toks in completed]

In [ ]:
print("Beam search vs greedy decoding:")
print("=" * 65)

for digits in test_numbers:
    src_tokens = torch.tensor([[d + 1 for d in digits] + [SRC_PAD] * (5 - len(digits))])
    expected_words = [TGT_TOKEN_NAMES[WORD_TOKENS[d]] for d in digits]

    # Greedy
    greedy_result = greedy_decode(model, src_tokens, max_len=10)
    greedy_words = [TGT_TOKEN_NAMES.get(t, f"?{t}") for t in greedy_result if t != TGT_EOS]

    # Beam search
    beam_results = beam_search(model, src_tokens, beam_width=4, max_len=10)

    print(f"Input: {digits}  |  Expected: {' '.join(expected_words)}")
    print(f"  Greedy: {' '.join(greedy_words)}")
    for rank, (log_prob, tokens) in enumerate(beam_results[:3]):
        words = [TGT_TOKEN_NAMES.get(t, f"?{t}") for t in tokens if t != TGT_EOS]
        print(f"  Beam {rank+1} (log_p={log_prob:.3f}): {' '.join(words)}")
    print()

In [ ]:
# Compare beam search accuracy vs greedy on the full validation set
num_correct_beam = 0

for i in range(len(val_src)):
    src_sample = val_src[i].unsqueeze(0)
    beam_results = beam_search(model, src_sample, beam_width=4, max_len=10)

    best_tokens = beam_results[0][1]  # best beam
    expected = [t for t in val_tgt_out[i].tolist() if t != -100]

    if best_tokens == expected:
        num_correct_beam += 1

beam_acc = num_correct_beam / len(val_src)

print(f"Full-sequence accuracy (greedy):      {seq_acc_ar:.4f}")
print(f"Full-sequence accuracy (beam w=4):    {beam_acc:.4f}")

---
## 8. Visualize Cross-Attention

The cross-attention weights reveal which source (input) tokens each
decoder (output) token attends to. For number-to-words, we expect
a roughly diagonal pattern: the first output word should attend to
the first input digit, and so on.

In [ ]:
def get_cross_attention_maps(model, src, tgt):
    """Run forward pass and collect cross-attention weights from all decoder layers.

    Args:
        src: (1, src_len)
        tgt: (1, tgt_len)

    Returns:
        List of tensors per decoder layer, each (num_heads, tgt_len, src_len)
    """
    model.eval()
    with torch.no_grad():
        _ = model(src.to(device), tgt.to(device))

    cross_attn_maps = []
    for layer in model.decoder.layers:
        weights = layer.cross_attn.attn_weights[0].cpu()  # (num_heads, tgt_len, src_len)
        cross_attn_maps.append(weights)
    return cross_attn_maps

In [ ]:
# Pick a sample to visualize
viz_digits = [3, 1, 4, 1, 5]
viz_src = torch.tensor([[d + 1 for d in viz_digits]])
viz_tgt_tokens = [TGT_BOS] + [WORD_TOKENS[d] for d in viz_digits]
viz_tgt = torch.tensor([viz_tgt_tokens])

src_labels = [str(d) for d in viz_digits]
tgt_labels = [TGT_TOKEN_NAMES[t] for t in viz_tgt_tokens]

print(f"Source: {src_labels}")
print(f"Target: {tgt_labels}")

cross_maps = get_cross_attention_maps(model, viz_src, viz_tgt)
print(f"Number of decoder layers: {len(cross_maps)}")
print(f"Cross-attention shape per layer: {cross_maps[0].shape}")

### 8.1 Cross-attention by head and layer

In [ ]:
num_dec_layers = len(cross_maps)
num_heads = cross_maps[0].shape[0]

fig, axes = plt.subplots(num_dec_layers, num_heads, figsize=(3.5 * num_heads, 3 * num_dec_layers))
if num_dec_layers == 1:
    axes = axes.reshape(1, -1)

for layer_idx in range(num_dec_layers):
    for head_idx in range(num_heads):
        ax = axes[layer_idx, head_idx]
        weights = cross_maps[layer_idx][head_idx].numpy()
        im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1, aspect="auto")

        ax.set_xticks(range(len(src_labels)))
        ax.set_xticklabels(src_labels, fontsize=8)
        ax.set_yticks(range(len(tgt_labels)))
        ax.set_yticklabels(tgt_labels, fontsize=8)
        ax.set_title(f"Layer {layer_idx + 1}, Head {head_idx + 1}", fontsize=9)

        if head_idx == 0:
            ax.set_ylabel("Decoder token")
        if layer_idx == num_dec_layers - 1:
            ax.set_xlabel("Encoder token")

fig.suptitle("Cross-Attention: Which Input Token Does Each Output Token Attend To?",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 8.2 Averaged cross-attention (last layer)

In [ ]:
avg_cross_attn = cross_maps[-1].mean(dim=0).numpy()  # (tgt_len, src_len)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(avg_cross_attn, cmap="Blues", vmin=0, aspect="auto")

ax.set_xticks(range(len(src_labels)))
ax.set_xticklabels(src_labels, fontsize=10)
ax.set_yticks(range(len(tgt_labels)))
ax.set_yticklabels(tgt_labels, fontsize=10)
ax.set_xlabel("Source (digits)", fontsize=11)
ax.set_ylabel("Target (words)", fontsize=11)
ax.set_title("Average Cross-Attention (Last Decoder Layer)", fontsize=12)
fig.colorbar(im, ax=ax, shrink=0.8)

# Annotate cells with values
for i in range(avg_cross_attn.shape[0]):
    for j in range(avg_cross_attn.shape[1]):
        val = avg_cross_attn[i, j]
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=color)

plt.tight_layout()
plt.show()

### 8.3 Cross-attention for multiple examples

In [ ]:
examples_for_viz = [
    [7, 2],
    [9, 0, 5, 8],
    [1, 2, 3],
]

fig, axes = plt.subplots(1, len(examples_for_viz), figsize=(5 * len(examples_for_viz), 4))

for ax_idx, digits in enumerate(examples_for_viz):
    ax = axes[ax_idx]
    ex_src = torch.tensor([[d + 1 for d in digits] + [SRC_PAD] * (5 - len(digits))])
    ex_tgt_tokens = [TGT_BOS] + [WORD_TOKENS[d] for d in digits]
    ex_tgt = torch.tensor([ex_tgt_tokens])

    ex_src_labels = [str(d) for d in digits] + ["[PAD]"] * (5 - len(digits))
    ex_tgt_labels = [TGT_TOKEN_NAMES[t] for t in ex_tgt_tokens]

    ex_cross_maps = get_cross_attention_maps(model, ex_src, ex_tgt)
    avg_attn = ex_cross_maps[-1].mean(dim=0).numpy()

    im = ax.imshow(avg_attn, cmap="Blues", vmin=0, aspect="auto")
    ax.set_xticks(range(len(ex_src_labels)))
    ax.set_xticklabels(ex_src_labels, fontsize=9)
    ax.set_yticks(range(len(ex_tgt_labels)))
    ax.set_yticklabels(ex_tgt_labels, fontsize=9)
    ax.set_xlabel("Source")
    if ax_idx == 0:
        ax.set_ylabel("Target")
    ax.set_title(f"Input: {digits}", fontsize=10)

fig.suptitle("Cross-Attention Patterns Across Examples (Last Layer, Head Average)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Decoder Self-Attention Visualization

For completeness, we also inspect the causal self-attention patterns
within the decoder. These show how each decoder position attends to
previous decoder positions.

In [ ]:
def get_decoder_self_attention_maps(model, src, tgt):
    """Collect decoder self-attention weights."""
    model.eval()
    with torch.no_grad():
        _ = model(src.to(device), tgt.to(device))

    self_attn_maps = []
    for layer in model.decoder.layers:
        weights = layer.self_attn.attn_weights[0].cpu()
        self_attn_maps.append(weights)
    return self_attn_maps


dec_self_maps = get_decoder_self_attention_maps(model, viz_src, viz_tgt)

fig, axes = plt.subplots(1, NUM_HEADS, figsize=(3.5 * NUM_HEADS, 3.5))

for head_idx in range(NUM_HEADS):
    ax = axes[head_idx]
    weights = dec_self_maps[-1][head_idx].numpy()
    im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1, aspect="auto")

    ax.set_xticks(range(len(tgt_labels)))
    ax.set_xticklabels(tgt_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(tgt_labels)))
    ax.set_yticklabels(tgt_labels, fontsize=8)
    ax.set_title(f"Head {head_idx + 1}", fontsize=10)

fig.suptitle("Decoder Self-Attention (Last Layer, Causal)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Analysis: Accuracy by Sequence Length

Longer sequences are harder. We break down the autoregressive
accuracy by the number of digits in the source.

In [ ]:
# Compute non-pad length for each validation source
val_lengths = (val_src != SRC_PAD).sum(dim=1).tolist()

length_correct = {}
length_total = {}

for i in range(len(val_src)):
    src_sample = val_src[i].unsqueeze(0)
    generated = greedy_decode(model, src_sample, max_len=10)
    expected = [t for t in val_tgt_out[i].tolist() if t != -100]

    seq_len = val_lengths[i]
    length_total[seq_len] = length_total.get(seq_len, 0) + 1
    if generated == expected:
        length_correct[seq_len] = length_correct.get(seq_len, 0) + 1

lengths_sorted = sorted(length_total.keys())
accs_by_length = [length_correct.get(l, 0) / length_total[l] for l in lengths_sorted]
counts_by_length = [length_total[l] for l in lengths_sorted]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(lengths_sorted, accs_by_length, color="steelblue", width=0.6)
for l, acc in zip(lengths_sorted, accs_by_length):
    ax1.text(l, acc + 0.02, f"{acc:.2f}", ha="center", fontsize=9)
ax1.set_xlabel("Source sequence length")
ax1.set_ylabel("Full-sequence accuracy")
ax1.set_title("Accuracy by Sequence Length (Autoregressive)")
ax1.set_ylim(0, 1.15)
ax1.set_xticks(lengths_sorted)

ax2.bar(lengths_sorted, counts_by_length, color="gray", width=0.6)
ax2.set_xlabel("Source sequence length")
ax2.set_ylabel("Number of samples")
ax2.set_title("Validation Set Length Distribution")
ax2.set_xticks(lengths_sorted)

plt.tight_layout()
plt.show()

---
## 11. Encoder Representations

We can inspect how the encoder represents different digits by
looking at the cosine similarity between encoder output vectors.

In [ ]:
# Encode single-digit sequences and collect representations
model.eval()
digit_representations = []

with torch.no_grad():
    for d in range(10):
        src = torch.tensor([[d + 1] + [SRC_PAD] * 4], device=device)
        enc_out, _ = model.encode(src)
        # Take the representation of the first (non-pad) position
        digit_representations.append(enc_out[0, 0].cpu())

digit_reps = torch.stack(digit_representations)  # (10, d_model)
digit_reps_normed = digit_reps / digit_reps.norm(dim=-1, keepdim=True)
cosine_sim = torch.matmul(digit_reps_normed, digit_reps_normed.T)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cosine_sim.numpy(), cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(10))
ax.set_xticklabels([str(d) for d in range(10)])
ax.set_yticks(range(10))
ax.set_yticklabels([str(d) for d in range(10)])
ax.set_xlabel("Digit")
ax.set_ylabel("Digit")
ax.set_title("Cosine Similarity of Encoder Representations")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

---
## Summary

In this notebook we built a full encoder-decoder transformer from scratch:

1. **Encoder** -- bidirectional self-attention over the source sequence, producing context-aware representations of each input token.

2. **Decoder** -- causal self-attention over previously generated tokens, plus cross-attention that queries the encoder output. This is the key architectural difference from decoder-only models.

3. **Full encoder-decoder model** -- wiring encoder, decoder, and output projection with proper masking for padding and causality.

4. **Number-to-words task** -- a toy but genuine seq2seq problem where the input and output vocabularies differ and the model must learn a token-level mapping.

5. **Teacher forcing vs autoregressive decoding** -- teacher forcing uses ground-truth tokens as decoder input, while autoregressive decoding feeds the model's own predictions. The gap between them reveals exposure bias.

6. **Beam search** -- maintaining multiple candidate sequences to find globally better outputs than greedy decoding.

7. **Cross-attention visualization** -- the cross-attention weights directly show which source tokens influence each output token, confirming the model learns the correct digit-to-word alignment.

This architecture (with variations) underlies T5, BART, mBART, and the original "Attention Is All You Need" transformer.